In [1]:
# Cell 2: Mount Google Drive (stores reference genome permanently)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 8: One‑hot encode, merge, shuffle, and flatten (keep in RAM for ML)
import numpy as np

def fasta_to_onehot(fasta_path, label):
    """Convert cleaned FASTA to one‑hot array and label vector."""
    sequences = []
    with open(fasta_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                sequences.append(line)

    n_samples = len(sequences)
    seq_array = np.array([list(seq) for seq in sequences], dtype='U1')
    onehot = np.zeros((n_samples, 101, 4), dtype=np.int8)

    # Vectorized channel assignment
    onehot[seq_array == 'A', 0] = 1
    onehot[seq_array == 'C', 1] = 1
    onehot[seq_array == 'G', 2] = 1
    onehot[seq_array == 'T', 3] = 1

    y = np.full(n_samples, label, dtype=np.int8)
    return onehot, y

# --- Main ---
base_dir = "/content/drive/MyDrive/ML_Project/data"
pos_clean = f"{base_dir}/sp1_positive_101bp_clean.fasta"
neg_clean = f"{base_dir}/sp1_negative_101bp_clean.fasta"

print("Encoding positive samples (label=1)...")
X_pos, y_pos = fasta_to_onehot(pos_clean, label=1)

print("Encoding negative samples (label=0)...")
X_neg, y_neg = fasta_to_onehot(neg_clean, label=0)

# Merge and shuffle
X = np.concatenate([X_pos, X_neg], axis=0)
y = np.concatenate([y_pos, y_neg], axis=0)

np.random.seed(42)
indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

print("\n=== Data ready for Deep Learning (3D tensor) ===")
print(f"X shape: {X.shape}   # (samples, 101, 4)")
print(f"y shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Flatten for classical ML algorithms
X_flat = X.reshape(X.shape[0], -1)   # (N, 404)
print("\n=== Data ready for Classical ML (2D matrix) ===")
print(f"X_flat shape: {X_flat.shape}")
print("✅ Variables 'X_flat' and 'y' are now in RAM. Ready for Scikit‑learn.")

Encoding positive samples (label=1)...
Encoding negative samples (label=0)...

=== Data ready for Deep Learning (3D tensor) ===
X shape: (47856, 101, 4)   # (samples, 101, 4)
y shape: (47856,)
Class distribution: [23316 24540]

=== Data ready for Classical ML (2D matrix) ===
X_flat shape: (47856, 404)
✅ Variables 'X_flat' and 'y' are now in RAM. Ready for Scikit‑learn.


In [4]:
# Cell 9: Classical ML Baseline Comparison
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import pandas as pd
import numpy as np
import time

# 1. Train/test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class dist: {np.bincount(y_train)}")
print(f"Test class dist:  {np.bincount(y_test)}\n")

# 2. Define models for comparison
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        n_jobs=-1,
        random_state=42
    )
}

# 3. Train and evaluate
results = []

for name, model in models.items():
    print(f"[Training] {name}...")
    start_time = time.time()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    elapsed = time.time() - start_time

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "AUC-ROC": round(auc, 4),
        "Time (s)": round(elapsed, 2)
    })

    print(f"  -> Accuracy: {acc:.4f}, AUC: {auc:.4f}, Time: {elapsed:.2f}s")

# 4. Display comparison table
df_results = pd.DataFrame(results).sort_values(by="AUC-ROC", ascending=False)
print("\n=== Classical ML Model Comparison ===")
print(df_results.to_string(index=False))

# 5. Detailed report for best model
best_model_name = df_results.iloc[0]['Model']
best_model = models[best_model_name]
print(f"\n=== Detailed Classification Report: {best_model_name} ===")
print(classification_report(y_test, best_model.predict(X_test)))

Train shape: (38284, 404), Test shape: (9572, 404)
Train class dist: [18652 19632]
Test class dist:  [4664 4908]

[Training] Logistic Regression...
  -> Accuracy: 0.8660, AUC: 0.9426, Time: 2.37s
[Training] Decision Tree...
  -> Accuracy: 0.7113, AUC: 0.7724, Time: 2.07s
[Training] Random Forest...
  -> Accuracy: 0.8715, AUC: 0.9414, Time: 12.97s

=== Classical ML Model Comparison ===
              Model  Accuracy  AUC-ROC  Time (s)
Logistic Regression    0.8660   0.9426      2.37
      Random Forest    0.8715   0.9414     12.97
      Decision Tree    0.7113   0.7724      2.07

=== Detailed Classification Report: Logistic Regression ===
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4664
           1       0.87      0.87      0.87      4908

    accuracy                           0.87      9572
   macro avg       0.87      0.87      0.87      9572
weighted avg       0.87      0.87      0.87      9572

